# Market Making Fundamentals

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: Bid/ask, RFQs, side conventions, inventory and cash.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


                timestamp ticker   mid  market_bid  market_ask
2026-01-02 09:30:00+00:00   ALFA 100.0        99.9       100.1
2026-01-02 09:30:00+00:00   BETA 100.0        99.9       100.1


## Two-sided price

Given a positive mid and full spread fraction, return (bid, ask). Example 100 and 0.02 -> (99,101). Reject invalid inputs.


In [ ]:
def two_sided(mid: float, spread: float) -> tuple[float, float]:
    # TODO
    if mid < 0 or mid.isinfinite() or spread < 0 or spread.isinfinite():
        raise ValueError("Mid and spread must be non-negative and finite.")
    bid = mid - spread * 100 / 2
    ask = mid + spread *100 / 2
    return bid, ask

print(two_sided(100, 0.02))
print(two_sided(100, 0.1))  # should raise ValueError

(99.0, 101.0)
(95.0, 105.0)


## Join RFQs to prices

Return an RFQ DataFrame enriched with `mid`, matching timestamp and ticker. Preserve all RFQs. Raise ValueError if any match is missing or duplicated.


In [6]:
def enrich_rfqs(rfqs: pd.DataFrame, market: pd.DataFrame) -> pd.DataFrame:
    # TODO
    if market[['timestamp', 'ticker']].duplicated().any():
        raise ValueError("Market data contains duplicate timestamp-ticker pairs.")
    enriched = rfqs.merge(market[['timestamp', 'ticker', 'mid']], on=['timestamp', 'ticker'], how='left', validate='many_to_one')
    if enriched['mid'].isna().any():
        raise ValueError("missing price data for some RFQs")
    return enriched

joined = enrich_rfqs(requests, prices)
print(requests.head(2).to_string(index=False))
print(joined.head(2).to_string(index=False))


                timestamp ticker  quantity
2026-01-02 09:30:00+00:00  GAMMA        10
2026-01-02 09:31:00+00:00   ALFA        50
                timestamp ticker  quantity        mid
2026-01-02 09:30:00+00:00  GAMMA        10 100.000000
2026-01-02 09:31:00+00:00   ALFA        50  98.806249


## Maker cash update

Return (new_inventory, new_cash). A maker buy of q at p adds q shares and spends qp; a maker sell subtracts q shares and receives qp. Fee rate is applied to notional on either side.


In [ ]:
def apply_fill(inventory: int, cash: float, side: str, q: int, p: float, fee_rate: float = 0.0) -> tuple[int, float]:
    # TODO
    if q < 0 or p < 0 or fee_rate < 0 or fee_rate > 1:
        raise ValueError("Wrong values for q, p, or fee_rate.")
    new_inventory = inventory
    new_cash = cash
    if side == 'buy':
        new_inventory += q
        new_cash -= q * p * (1 + fee_rate)
    elif side == 'sell':
        new_inventory -= q
        new_cash += q * p * (1 - fee_rate)
    else:
        raise ValueError("Side must be 'buy' or 'sell'.")
    return new_inventory, new_cash

print(apply_fill(0, 1000, 'buy', 2, 100, 0.01))
print(apply_fill(0, 1000, 'sell', 2, 100, 0.01))

(2, 798.0)
(-2, 1198.0)


## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.